In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from matplotlib.ticker import ScalarFormatter
from spectrum_component_analyser.spectral_component import spectral_component
from spectrum_component_analyser.mcmc.simulated_spectra import get_simulated_spectra
from spectrum_component_analyser.mcmc.constrained_helper import ConstrainedMCMCHelper
from phoenix_grid_creator.spectral_grid import spectral_grid

from constants import *
from astropy import units as u

from phoenix_grid_creator.spectral_grid import spectral_grid
from spectrum_component_analyser.phoenix_spectrum import phoenix_spectrum
from spectrum_component_analyser.chi_squared_minimisation import ChiHelper, get_chi_r

# both actual global constants
NUMBER_OF_PARAMETERS : int = 4 # weight, t, f, l

fits_file_paths = list(Path(package_path / "raw_phoenix_spectra").rglob("*.fits"))

In [ ]:
snrs = np.logspace(-2, 6, 10)
print(snrs)

true_feh = 0.0 * u.dex
true_logg = 4.5 * u.dex

true_components = [
    spectral_component(3800 * u.K, true_feh, true_logg, 0.85),
    spectral_component(3300 * u.K, true_feh, true_logg, 0.15)
]

results = []

resolution = 0.3 * u.nm
observational_wavelengths = np.arange(0.8, 5.5, resolution.to(u.um).value) * u.um
print(observational_wavelengths)
spec_grid = spectral_grid.from_local_raw(fits_file_paths, False, observational_wavelengths)
spec_grid.Wavelengths = spec_grid.Wavelengths.to(u.um)
spec_grid.Fluxes *= u.Jy

for snr in snrs:
    print(f"SNR = {snr}")
    _, _, obs = get_simulated_spectra(spec_grid, true_components, snr=snr)
    
    chi_r = get_chi_r(len(true_components), NUMBER_OF_PARAMETERS, obs, spec_grid)
    
    # should be done by spectral grid class or the mcmchelper classes
    parameter_bounds = [
        (.0, 2.),
        (np.min(spec_grid.T_effs.value), np.max(spec_grid.T_effs.value)),
        (np.min(spec_grid.FeHs.value), np.max(spec_grid.FeHs.value)),
        (np.min(spec_grid.Log_gs.value), np.max(spec_grid.Log_gs.value)),
    ]

    mcmc = ConstrainedMCMCHelper(
        parameter_bounds=parameter_bounds,
        number_of_components=len(true_components),
        observed_spectrum=obs,
        spec_grid=spec_grid,
        n_steps=1500,
        n_walkers=int(2.5 * len(true_components) * NUMBER_OF_PARAMETERS)
    )
    
    sampler, samples = mcmc.run(chi_r)
    results.append(np.percentile(samples, [16, 50, 84], axis=0))

medians = np.array([r[1] for r in results])
lower_err = medians - np.array([r[0] for r in results])
upper_err = np.array([r[2] for r in results]) - medians

In [ ]:
import corner
import emcee


def plot_corner(mcmc : ConstrainedMCMCHelper, sampler : emcee.EnsembleSampler, samples : np.ndarray, true_components : list[spectral_component]):
    labels = []
    for i in range(mcmc.number_of_components):
        labels += [f"$f_{i+1}$", f"$T_{i+1}$ [K]"]
    labels += ["[Fe/H] [dex]", r"$\log g$ [dex]"]

    main_fill_color = "#FF8B00"
    quantile_line_color = "#AC8DD9"

    # fig, axes = plt.subplots(samples.shape[1], samples.shape[1], figsize=(16, 16))
    
    fontsize = 12
    plt.rc('xtick', labelsize=fontsize - 2)
    plt.rc('ytick', labelsize=fontsize - 2)

    true_params = []
    for c in true_components:
        true_params += [c.Weight, c.T_eff.value]
    true_params += [true_feh.value, true_logg.value]
    print(true_feh)
    print(true_logg)
    
    true_params = np.array(true_params, dtype=float)

    truth_color = "black"

    fig = corner.corner(
        samples,
        truths=true_params,
        truth_color=truth_color,
        # fig=fig,
        labels=labels, 
        quantiles=[0.16, 0.5, 0.84],
        show_titles=True,
        # --- Neatness Settings ---
        color=main_fill_color,
        plot_datapoints=False,         # Removes the scatter points
        plot_density=False,            # Removes the low-res "square" histogram
        fill_contours=True,         # Keeps it looking minimal
        levels=(1 - np.exp(-0.5 * np.arange(1, 4)**2)), # Standard 1, 2, 3 sigma
        contour_kwargs={"colors": [main_fill_color], "linewidths": 1.0},
        hist_kwargs={
            "color": "black",             # This sets the 'edgecolor' when histtype='step'
            "fill": True,                 # Enables the fill
            "facecolor": main_fill_color, # The color inside the histograms
            "edgecolor": "black",         # The outline color
            "linewidth": 1.5,
            "alpha": 0.5                  # Slight transparency often looks cleaner
        },
        title_kwargs={"fontsize": fontsize},
        label_kwargs={"fontsize": fontsize},     
        )
    
    axes = np.array(fig.axes).reshape((samples.shape[1], samples.shape[1]))

    for i in range(samples.shape[1]):
        ax = axes[i, i] # Target the 1D histograms on the diagonal

        current_title = ax.get_title()
        if "T_" in labels[i]:
            value = current_title.split("=")[1]
            ax.set_title(r"$T_\text{eff}$ =" + f"{value} K", fontsize=fontsize)
        elif "Fe/H" in labels[i]:
            value = current_title.split("=")[1]
            ax.set_title(r"[Fe/H] =" + f"{value} dex", fontsize=fontsize)
        elif "log" in labels[i]:
            value = current_title.split("=")[1]
            ax.set_title(r"$\log g$ =" + f"{value} dex", fontsize=fontsize)
        
        # Corner usually plots quantile lines as 'Line2D' objects
        idx = 0
        for line in ax.get_lines():
            if line.get_color() == truth_color:
                continue
            if (idx == 1):
                line.set_linestyle("-")
            # only count the lines that aren't truth-lines
            idx += 1

            # line.set_color("#705CC8B0")
            # line.set_linestyle("--") # Optional: make them dashed
            # line.set_linewidth(2)     # Optional: make them thicker
    
    for ax in fig.axes:
        for line in ax.get_lines():
            if line.get_color() == truth_color:
                line.set_linewidth(1)
                line.set_markersize(5)
                line.set_zorder(10)

    plt.rcParams['xtick.direction'] = 'in'
    plt.rcParams['ytick.direction'] = 'in'
    # Optional: makes ticks appear on all sides
    plt.rcParams['xtick.top'] = True
    plt.rcParams['ytick.right'] = True

    # fig.suptitle("Posterior distributions for a simulated M dwarf", fontsize=20)
    fig.subplots_adjust(top=0.93)
    plt.show()

    # caterpillar
    fig, axes = plt.subplots(mcmc.ndim, figsize=(10, 7), sharex=True)
    chain = sampler.get_chain()
    for i in range(mcmc.ndim):
        ax = axes[i]
        ax.plot(chain[:, :, i], "k", alpha=0.3)
        ax.set_xlim(0, len(chain))
        ax.set_ylabel(labels[i])
        ax.yaxis.set_label_coords(-0.1, 0.5)

    axes[-1].set_xlabel("Step number")
    # plt.tight_layout()
    plt.show()

In [ ]:
plot_corner(mcmc, sampler, samples, true_components)

%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

In [ ]:
print(medians)

In [ ]:
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
fontsize=14
comp_colors = ['royalblue', 'forestgreen']

for i in range(len(true_components)):
    t_true = true_components[i].T_eff.value
    w_true = true_components[i].Weight
    
    w_idx = i * 2
    t_idx = i * 2 + 1

    print(snr)
    
    ax1.errorbar(snrs, medians[:, t_idx] - t_true, 
                 yerr=[lower_err[:, t_idx], upper_err[:, t_idx]],
                 fmt='o', color=comp_colors[i], capsize=5,
                 label=f'Component {i+1} ({t_true}K)')
    
    ax2.errorbar(snrs, medians[:, w_idx], 
                 yerr=[lower_err[:, w_idx], upper_err[:, w_idx]],
                 fmt='s', color=comp_colors[i], capsize=5)
                #  label=f'Component {i+1} (Truth: {w_true})')
    ax2.axhline(w_true, color=comp_colors[i], alpha=0.99, linestyle=':', label="$f_"+f"{i+1}"+f"={w_true}$ (ground truth)")

# real life resolutions
# ax1.axvline(2.2*u.um/150)
# ax1.axvline(0.8*u.um/150)
# ax1.axvline(5*u.um/1000)
# ax1.axvline(2*u.um/3500)
# ax1.axvline(0.8*u.um/150)
print(resolution)
print(snrs)

ax1.axhline(0, color='red', alpha=0.5)
ax1.set_ylabel(r"$T_{\mathrm{found}} - T_{\mathrm{true}}$ [K]",fontsize=fontsize+2)
ax1.legend(fontsize=fontsize, loc='lower right')
ax1.set_ylim(-850)

ax2.set_ylabel("$f$", fontsize=fontsize+2)
ax2.set_ylim(-0.08, 1.08)
ax2.set_xlabel(r"SNR", fontsize=fontsize)
ax2.set_xscale('log')
# ax2.xaxis.set_major_formatter(ScalarFormatter())
leg = ax2.legend(fontsize=fontsize, loc='upper right')
for line in leg.get_lines():
    line.set_linewidth(2.0)
for ax in [ax1, ax2]:
    ax.grid(which='both', alpha=0.6)
    ax.invert_xaxis()
    ax.tick_params(axis='both', which='major', labelsize=12, length=8, width=1.5)

plt.tight_layout()
plt.savefig("temp.svg", format="svg", bbox_inches='tight')
plt.show()